In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random

%matplotlib inline

In [ ]:
root = Path('..')
data_path = root / 'data' / 'names.txt'
words = open(data_path).read().splitlines()

print(f'Words count: {len(words)}')

In [ ]:
# build vocabulary of characters and mappings to/from ints

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

In [ ]:
# build dataset split into train / val / test

def build_dataset(words, block_size=3):
    X, Y = [], []

    for w in words:

        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, X.dtype, Y.shape, Y.dtype)
    return X, Y

X, Y = build_dataset(words)

s1 = int(0.8 * len(X))
s2 = int(0.9 * len(X))

X_train, Y_train = X[:s1], Y[:s1]
X_val, Y_val = X[s1:s2], Y[s1:s2]
X_test, Y_test = X[s2:], Y[s2:]

print(f'train: {len(X_train)} val: {len(X_val)} test: {len(X_test)}')

In [ ]:
n_embed = 10
n_hidden = 200
vocab_size = len(stoi)
block_size = 3
g = torch.Generator().manual_seed(42)
C = torch.randn((vocab_size, n_embed), generator=g) # 27 characters, increased embedding size to 10 dimensions
print(C.shape)

W1 = torch.randn((n_embed * block_size, n_hidden), generator=g) * (5/3) / ((n_embed * block_size) ** 0.5)
# b1 = torch.randn(n_hidden, generator=g) * 0.025
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.05
b2 = torch.randn(vocab_size, generator=g) * 0

bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, W2, b2, bngain, bnbias]

total_params = sum(p.nelement() for p in parameters) # total number of parameters
print(f'Total parameters: {total_params}')

for p in parameters: p.requires_grad = True # enable gradients

In [ ]:
EPOCHS = 100_000
BATCH_SIZE = 64

LR = 0.135

lri = []
lossi = []
stepi = [] # let's also track the training steps

In [ ]:
# Training loop
# use cross-entropy loss and batch norm

for epoch in range(EPOCHS):

    # Minibatch
    indices = torch.randint(0, X_train.shape[0], (BATCH_SIZE,), generator=g)
    Xb, Yb = X_train[indices], Y_train[indices] # batch X, Y

    # Forward pass
    emb = C[Xb]
    embcat = emb.view(emb.shape[0], -1)

    # Linear layer
    hpreact = embcat @ W1 # + b1 # no bias, as batch norm will handle it with bnbias

    # Batch norm layer
    bnmeani = hpreact.mean(0, keepdim=True)
    bnstdi = hpreact.std(0, keepdim=True)
    hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias # batch norm
    with torch.no_grad():
        bnmean_running = 0.99 * bnmean_running + 0.01 * bnmeani
        bnstd_running = 0.99 * bnstd_running + 0.01 * bnstdi

    # Non-linearity
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    # Log loss
    if epoch % 5_000 == 0: print(f'epoch: {epoch} loss: {loss.item()}')

    # Backward pass
    for p in parameters: p.grad = None
    loss.backward()

    # Gradient descent step 
    lr = LR if epoch < 50_000 else LR / 10 # weight decay after 125k steps
    for p in parameters: p.data += -lr * p.grad

    # Track loss
    lri.append(lr)
    lossi.append(loss.log10().item())
    stepi.append(epoch)

In [ ]:
# plt.plot(stepi, lossi)

In [ ]:
# plt.hist(h.view(-1).tolist(), 50) # histogram of hidden layer activations

In [ ]:
# plt.hist(hpreact.view(-1).tolist(), 50) # histogram of hidden layer pre-activations

In [ ]:
# calibrate the batch norm at the end of training

with torch.no_grad():
  # pass the training set through
  emb = C[Xb]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 # + b1
  # measure the mean/std over the entire training set
  bnmean = hpreact.mean(0, keepdim=True)
  bnstd = hpreact.std(0, keepdim=True)

In [ ]:
plt.figure(figsize=(20, 10))
plt.imshow(h.abs() > 0.99, cmap='gray', interpolation='nearest') # visualize the activations of the hidden layer

In [ ]:
@torch.no_grad()
def split_loss(split):
    x, y = {
        'train': (X_train, Y_train),
        'val': (X_val, Y_val),
        'test': (X_test, Y_test)
    }[split]
    emb = C[x]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 # + b1
    hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    print(f'{split} loss: {loss.item()}')

split_loss('train')
split_loss('val')
split_loss('test')

In [ ]:
# Rewriting lecture code to PyTorch like implementation

class Linear:

    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out), generator=g) # / fan_in ** 0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None: self.out += self.bias
        return self.out
    
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])
    
class BatchNorm1d:

    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True

        # Parameters (trainable)
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

        # Buffers (trained with a running 'momentum update')
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):

        # Forward pass
        if self.training:
            xmean = x.mean(0, keepdim=True) # batch mean
            xvat = x.var(0, unbiased=True, keepdim=True) # batch variance
        else:
            xmean = self.running_mean
            xvat = self.running_var
        
        xhat = (x - xmean) / torch.sqrt(xvat + self.eps) # normalize
        self.out = self.gamma * xhat + self.beta

        # Update the buffers (inference mode)
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvat
        
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

class Tanh:

    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
        return []

class ReLU:
    
    def __call__(self, x):
        self.out = x.clamp_min(0)
        return self.out

    def parameters(self):
        return []

n_embed = 10 # the dimensionality of the  character embeddings vector
n_hidden = 100 # the number of hidden units
g = torch.Generator().manual_seed(2147483647)

C = torch.randn((vocab_size, n_embed), generator=g) # character embeddings

# layers = [
#     Linear(n_embed * block_size, n_hidden), Tanh(),
#     Linear(            n_hidden, n_hidden), Tanh(),
#     Linear(            n_hidden, n_hidden), Tanh(),
#     Linear(            n_hidden, n_hidden), Tanh(),
#     Linear(            n_hidden, n_hidden), Tanh(),
#     Linear(            n_hidden, vocab_size),
# ]

# layers = [
#     Linear(n_embed * block_size, n_hidden),     BatchNorm1d(n_hidden),   Tanh(),
#     Linear(            n_hidden, n_hidden),     BatchNorm1d(n_hidden),   Tanh(),
#     Linear(            n_hidden, n_hidden),     BatchNorm1d(n_hidden),   Tanh(),
#     Linear(            n_hidden, n_hidden),     BatchNorm1d(n_hidden),   Tanh(),
#     Linear(            n_hidden, n_hidden),     BatchNorm1d(n_hidden),   Tanh(),
#     Linear(            n_hidden, vocab_size),   BatchNorm1d(vocab_size),
# ]

layers = [
    Linear(n_embed * block_size, n_hidden),     BatchNorm1d(n_hidden),   ReLU(),
    Linear(            n_hidden, n_hidden),     BatchNorm1d(n_hidden),   ReLU(),
    Linear(            n_hidden, n_hidden),     BatchNorm1d(n_hidden),   ReLU(),
    Linear(            n_hidden, n_hidden),     BatchNorm1d(n_hidden),   ReLU(),
    Linear(            n_hidden, n_hidden),     BatchNorm1d(n_hidden),   ReLU(),
    Linear(            n_hidden, vocab_size),   BatchNorm1d(vocab_size),
]

with torch.no_grad():
    # Last layer, make less confident
    # layers[-1].weight *= 0.1 # linear layer
    layers[-1].gamma *= 0.1 # batch norm
    # all other layers, apply gain
    for layer in layers[:-1]:
        if isinstance(layer, Linear):
            layer.weight *= 1.0 # 5/3

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters))
for p in parameters: p.requires_grad = True

In [ ]:
EPOCHS = 250_000
BATCH_SIZE = 64
lossi = []
ud = [] # update to data ratio

for epoch in range(EPOCHS):

    # Minibatch
    ix = torch.randint(0, X_train.shape[0], (BATCH_SIZE,), generator=g)
    Xb, Yb = X_train[ix], Y_train[ix]

    # Forward pass
    emb = C[Xb]
    x = emb.view(emb.shape[0], -1)
    for layer in layers:
        x = layer(x)
    loss = F.cross_entropy(x, Yb)

    # Backward pass
    for layer in layers: layer.out.retain_grad()
    for p in parameters: p.grad = None
    loss.backward()

    # Update 
    lr = 1.25 # 0.1 if epoch < EPOCHS / 2 else 0.01
    for p in parameters: p.data -= lr * p.grad

    # Track stats
    if epoch % 5_000 == 0: print(f'epoch: {epoch}/{EPOCHS} loss: {loss.item():.3f}')
    lossi.append(loss.log10().item())
    with torch.no_grad():
        ud.append([(lr * p.grad.std() / p.data.std()).log10().item() for p in parameters])


In [ ]:
# Visualize the activation distributions
plt.figure(figsize=(20, 5))
legends = []
for i, layer in enumerate(layers[:-1]): # note: excluding the output layer
    if isinstance(layer, (Tanh, ReLU)):
        t = layer.out
        print(f'layer {i} ({layer.__class__.__name__:10}) mean: {t.mean():.2f} std: {t.std():.2f}, saturated: {(t.abs() > 0.97).float().mean() * 100:.2f}%')
        hy, hx = torch.histogram(t, density=True)
        plt.plot(hx[:-1].detach(), hy.detach())
        legends.append(f'layer {i} {layer.__class__.__name__}')
plt.legend(legends)
plt.title('Activation distributions')

In [ ]:
# Visualize the gradient distributions
plt.figure(figsize=(20, 5))
legends = []
for i, layer in enumerate(layers[:-1]): # note: excluding the output layer
    if isinstance(layer, (Tanh, ReLU)):
        t = layer.out.grad
        print(f'layer {i} ({layer.__class__.__name__:10}) mean: {t.mean():+f} std: {t.std():e}')
        hy, hx = torch.histogram(t, density=True)
        plt.plot(hx[:-1].detach(), hy.detach())
        legends.append(f'layer {i} {layer.__class__.__name__}')
plt.legend(legends)
plt.title('Gradient distributions')

In [ ]:
# Visualize the weight distributions
plt.figure(figsize=(20, 5))
legends = []
for i, p in enumerate(parameters): # note: excluding the output layer
    t = p.grad
    if p.ndim == 2:
        print(f'weight: {tuple(p.shape)} \t mean: {t.mean():+f} std: {t.std():e} grad:data ratio: {t.std() / p.std():f}')
        hy, hx = torch.histogram(t, density=True)
        plt.plot(hx[:-1].detach(), hy.detach())
        legends.append(f'{i} {tuple(p.shape)}')
plt.legend(legends)
plt.title('Weight distributions')

In [ ]:
# Visualize the update to data ratio
plt.figure(figsize=(20, 5))
legends = []
for i, p in enumerate(parameters):
    if p.ndim == 2:
        plt.plot([ud[j][i] for j in range(len(ud))])
        legends.append(f'param: {i}')
plt.plot([0, len(ud)], [-3, -3], 'k')
plt.legend(legends)